In [1]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK 04 — AUTOMATED HYPERPARAMETER OPTIMIZATION
# Marathi Mitra — Using Optuna for smarter HPO
#
# This notebook:
# → Explains Bayesian optimization vs manual search
# → Runs 3 Optuna trials (compute-limited demo)
# → Compares with manual experiment results
# → Shows what full search would find
#
# Requires: GPU (T4 minimum)
# ═══════════════════════════════════════════════════════════

print("🔍 Marathi Mitra — Automated Hyperparameter Optimization")
print("=" * 60)

🔍 Marathi Mitra — Automated Hyperparameter Optimization


In [2]:
!pip install -q optuna
!pip install -q transformers==4.44.0
!pip install -q datasets==2.20.0
!pip install -q peft==0.12.0
!pip install -q trl==0.9.6
!pip install -q bitsandbytes==0.45.3
!pip install -q accelerate==0.33.0

print("✅ Libraries installed!")

  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [53 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\samik\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\samik\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\samik\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to minimal
      info: setting default host triple t

✅ Libraries installed!


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + c:\Users\samik\Data-Science-and-Machine-Learning-Guides\.venv\Scripts\python.exe C:\Users\samik\AppData\Local\Temp\pip-install-owrzt9tt\numpy_3a22e18092b842dc8b3d6cbad0594c5d\vendored-meson\meson\meson.py setup C:\Users\samik\AppData\Local\Temp\pip-install-owrzt9tt\numpy_3a22e18092b842dc8b3d6cbad0594c5d C:\Users\samik\AppData\Local\Temp\pip-install-owrzt9tt\numpy_3a22e18092b842dc8b3d6cbad0594c5d\.mesonpy-qp_f29yq -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\samik\AppData\Local\Temp\pip-install-owrzt9tt\numpy_3a22e18092b842dc8b3d6cbad0594c5d\.mesonpy-qp_f29yq\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\samik\AppData\Local\Temp\pip-install-owrzt9tt\numpy_3a22e18092b842dc8b3d6cbad0594c5d
      Build dir: C:\Users\samik\AppData\Local\Temp\

In [4]:
# ── This cell is educational — explains what Optuna does ──

explanation = """
HOW OPTUNA WORKS
═══════════════════════════════════════════════════════════

Manual Search (what we did in 02_finetune.ipynb):
──────────────────────────────────────────────────
Experiment 1: lr=2e-4, epochs=5,  r=16  → Score: 8%
Experiment 2: lr=2e-4, epochs=25, r=16  → Score: 60%
Experiment 3: lr=1e-4, epochs=25, r=16  → Score: ?%
Experiment 4: lr=2e-4, epochs=25, r=32  → Score: ?%

Problem:
→ We chose values based on intuition
→ May miss the true optimal combination
→ 4 experiments = 4 data points only

Optuna (Bayesian Optimization):
────────────────────────────────
Trial 1: lr=3e-4, epochs=12, r=24  → Score: 45%
         ↓ Optuna learns: high lr + medium epochs = ok
Trial 2: lr=1e-4, epochs=20, r=16  → Score: 72%
         ↓ Optuna learns: low lr + more epochs = better
Trial 3: lr=8e-5, epochs=28, r=16  → Score: 81%
         ↓ Optuna learns: even lower lr + more epochs best
Trial 4: lr=6e-5, epochs=30, r=16  → Score: 85% ← smarter guess
...

Gets smarter with each trial
Finds optimal in fewer experiments than grid search
"""

print(explanation)


HOW OPTUNA WORKS
═══════════════════════════════════════════════════════════

Manual Search (what we did in 02_finetune.ipynb):
──────────────────────────────────────────────────
Experiment 1: lr=2e-4, epochs=5,  r=16  → Score: 8%
Experiment 2: lr=2e-4, epochs=25, r=16  → Score: 60%
Experiment 3: lr=1e-4, epochs=25, r=16  → Score: ?%
Experiment 4: lr=2e-4, epochs=25, r=32  → Score: ?%

Problem:
→ We chose values based on intuition
→ May miss the true optimal combination
→ 4 experiments = 4 data points only

Optuna (Bayesian Optimization):
────────────────────────────────
Trial 1: lr=3e-4, epochs=12, r=24  → Score: 45%
         ↓ Optuna learns: high lr + medium epochs = ok
Trial 2: lr=1e-4, epochs=20, r=16  → Score: 72%
         ↓ Optuna learns: low lr + more epochs = better
Trial 3: lr=8e-5, epochs=28, r=16  → Score: 81%
         ↓ Optuna learns: even lower lr + more epochs best
Trial 4: lr=6e-5, epochs=30, r=16  → Score: 85% ← smarter guess
...

Gets smarter with each trial
Finds opt

In [5]:
import torch
import json
import optuna
import urllib.request
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ── Load dataset ─────────────────────────────────────────
GITHUB_USERNAME = "ninadparab"
url = (
    f"https://raw.githubusercontent.com/"
    f"{GITHUB_USERNAME}/marathi-mitra/main/"
    f"data/vocabulary_dataset.json"
)

with urllib.request.urlopen(url) as response:
    raw_data = json.loads(response.read().decode("utf-8"))

hf_dataset = Dataset.from_list(raw_data)
print(f"✅ Dataset: {len(hf_dataset)} examples")

# ── Load tokenizer once ───────────────────────────────────
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
tokenizer  = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✅ Tokenizer loaded")

# ── Test words and expected ───────────────────────────────
TEST_WORDS = ["butterfly", "mother", "rain", "elephant", "school"]
EXPECTED = {
    "butterfly": "फुलपाखरू",
    "mother":    "आई",
    "rain":      "पाऊस",
    "elephant":  "हत्ती",
    "school":    "शाळा",
}
print(f"✅ Ready for optimization")

c:\Users\samik\Data-Science-and-Machine-Learning-Guides\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# ── Reuse same functions from 02_finetune.ipynb ──────────

def load_fresh_model():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    return prepare_model_for_kbit_training(m)


def generate(word, model, tokenizer):
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {word}

### Response:
"""
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=512,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


def get_format_score(model, tokenizer):
    """Score model on all 5 test words. Returns 0-100."""
    total = 0
    for word in TEST_WORDS:
        output   = generate(word, model, tokenizer)
        expected = EXPECTED[word]
        criteria = {
            "Has Marathi word":   expected in output,
            "Has pronunciation":  "How to say" in output,
            "Has sentence":       "Example sentence" in output,
            "Has fun fact":       "Fun Fact" in output,
            "Has emojis":         "🌟" in output and "📢" in output,
        }
        total += sum(criteria.values()) / len(criteria) * 100
    return total / len(TEST_WORDS)


print("✅ Helper functions ready")

In [ ]:
# ── This is the core of Optuna ────────────────────────────
# objective() is called once per trial
# Optuna controls what values it passes
# We return the score → Optuna learns from it

def objective(trial):
    # ── Optuna SUGGESTS hyperparameter values ────────────
    # It learns from previous trials to make smarter guesses

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5, 5e-4,    # search between these bounds
        log=True       # log scale (better for lr search)
    )

    epochs = trial.suggest_int(
        "epochs",
        10, 30         # search between 10 and 30 epochs
    )

    r = trial.suggest_categorical(
        "r",
        [8, 16, 32]    # only try these specific values
    )

    lora_alpha = r * 2  # always keep alpha = 2x rank

    print(f"\n{'─'*50}")
    print(f"Trial {trial.number + 1}")
    print(f"  learning_rate: {learning_rate:.2e}")
    print(f"  epochs:        {epochs}")
    print(f"  r:             {r}")
    print(f"  lora_alpha:    {lora_alpha}")
    print(f"{'─'*50}")

    # ── Train with suggested params ───────────────────────
    model = load_fresh_model()

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    training_args = SFTConfig(
        output_dir=f"./optuna_trial_{trial.number}",
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=learning_rate,
        fp16=True,
        logging_steps=10,
        max_seq_length=512,
        dataset_text_field="text",
        warmup_ratio=0.1,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=hf_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()

    final_loss = trainer.state.log_history[-1].get("train_loss", 99)
    print(f"\n  Final loss: {final_loss:.4f}")

    # ── Evaluate and return score ─────────────────────────
    score = get_format_score(model, tokenizer)
    print(f"  Format score: {score:.1f}%")

    # Free GPU memory before next trial
    del model
    torch.cuda.empty_cache()

    # ── Return score → Optuna maximizes this ─────────────
    return score


print("✅ Objective function defined")
print("   Optuna will search:")
print("   → learning_rate: 1e-5 to 5e-4 (log scale)")
print("   → epochs:        10 to 30")
print("   → r:             8, 16, or 32")

In [ ]:
# ── Create and run the study ──────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# Suppress Optuna's own logging for cleaner output
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("🔍 Starting Optuna Hyperparameter Search")
print("=" * 60)
print("Note: Running 3 trials due to compute constraints")
print("      Full search would use 20+ trials")
print("      Each trial: ~5-7 minutes on T4")
print("      Total estimated time: ~20 minutes")
print("=" * 60)

# Create study — direction="maximize" because higher score = better
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),  # reproducible
    study_name="marathi-mitra-hpo",
)

# Add manual experiment results so Optuna learns from them!
# This is called "warm starting" — very powerful technique
study.enqueue_trial({"learning_rate": 2e-4, "epochs": 5,  "r": 16})
study.enqueue_trial({"learning_rate": 2e-4, "epochs": 25, "r": 16})
study.enqueue_trial({"learning_rate": 1e-4, "epochs": 25, "r": 16})

# Run trials
# n_trials=3 for demo — increase to 20+ for real search
study.optimize(
    objective,
    n_trials=3,
    show_progress_bar=False,
)

print("\n✅ Optuna search complete!")

In [ ]:
# ── Show what Optuna found ────────────────────────────────
print("\n" + "═" * 60)
print("OPTUNA RESULTS")
print("═" * 60)

print(f"\nAll trials:")
print(f"{'Trial':<8} {'LR':<10} {'Epochs':<8} {'r':<6} {'Score'}")
print("─" * 50)

for trial in study.trials:
    print(
        f"{trial.number + 1:<8} "
        f"{trial.params['learning_rate']:<10.2e} "
        f"{trial.params['epochs']:<8} "
        f"{trial.params['r']:<6} "
        f"{trial.value:.1f}%"
    )

print("═" * 60)
print(f"\n🏆 Best trial: {study.best_trial.number + 1}")
print(f"   Score:  {study.best_value:.1f}%")
print(f"   Params:")
for key, val in study.best_params.items():
    print(f"     {key}: {val}")

In [ ]:
# ── Side by side comparison ───────────────────────────────
print("\n" + "═" * 60)
print("MANUAL vs OPTUNA COMPARISON")
print("═" * 60)

manual_results = {
    "Manual Exp1 (lr=2e-4, ep=5,  r=16)": 8.0,
    "Manual Exp2 (lr=2e-4, ep=25, r=16)": 60.0,
    "Manual Exp3 (lr=1e-4, ep=25, r=16)": None,  # fill in your result
    "Manual Exp4 (lr=2e-4, ep=25, r=32)": None,  # fill in your result
}

print("\nManual experiments:")
for name, score in manual_results.items():
    score_str = f"{score:.1f}%" if score else "pending"
    bar = "█" * int((score or 0) / 10) + "░" * (10 - int((score or 0) / 10))
    print(f"  {name:<40} {bar} {score_str}")

print(f"\nOptuna trials:")
for trial in study.trials:
    bar = "█" * int(trial.value / 10) + "░" * (10 - int(trial.value / 10))
    print(
        f"  Trial {trial.number + 1} "
        f"(lr={trial.params['learning_rate']:.0e}, "
        f"ep={trial.params['epochs']}, "
        f"r={trial.params['r']})"
        f"{'':5} {bar} {trial.value:.1f}%"
    )

print(f"\n{'═'*60}")
print(f"Best Manual:  {max(v for v in manual_results.values() if v):.1f}%")
print(f"Best Optuna:  {study.best_value:.1f}%")

In [ ]:
# ── Educational cell — no code to run ────────────────────
full_search_info = """
WHAT A FULL OPTUNA SEARCH WOULD DO
═══════════════════════════════════════════════════════════

With n_trials=20 on Colab Pro (A100 GPU):
→ Each trial: ~3 minutes
→ Total time: ~60 minutes
→ Cost: ~$1 of Colab Pro credits

Search space explored:
→ learning_rate: 1e-5 to 5e-4 (continuous)
→ epochs:        10 to 30 (integer)
→ r:             8, 16, 32 (categorical)

Optuna would likely find:
→ Optimal lr around 5e-5 to 1e-4
→ Optimal epochs around 25-35
→ Optimal r = 16 (our dataset is small)

Why 3 trials is enough for a demo:
→ Shows you understand the CONCEPT
→ Proves the code works
→ Demonstrates warm-starting with manual results
→ README note: "Full search needs more compute"

This is EXACTLY what real ML engineers do:
→ Run limited search to validate approach
→ Document what full search would need
→ Make informed compute vs quality tradeoffs
"""
print(full_search_info)